# ANNITIA Challenge — Prédiction de survie MASLD

**Auteur :** YEVI Mawuli Peniel Samuel — IFRI-UAC, Bénin  
**Challenge :** TRUSTII / ICAN — Hepatic Event & All-Cause Mortality Prediction  
**Score final OOF :** 0.8991  (C-index hépatique = 0.8865, C-index décès = 0.9284)

---

## Vue d'ensemble du pipeline

```
Raw data (1 676 patients × ≤22 visits × 18 features)
         │
         ├────────────────────────────────────────────┐
         ▼                                            ▼
  [Path A: XGBoost Cox]                  [Path B: K-Mamba SSM]  ← BEST
  Temporal feature engineering           Pure C implementation
  (slope, variability, ratios…)          MambaBlock × 2, state=16
  ~162 flat features                     MIMO rank=4, 4 parallel channels
  5-fold × 5 seeds                       5-fold × 3 seeds × 200 epochs
  OOF = 0.8823                           OOF = 0.8991  ★ SOUMIS
         │                                            │
         └──────────────────────┬─────────────────────┘
                                ▼
                     submission_final.csv
                 (trustii_id, risk_hepatic, risk_death)
```

## Ce notebook est auto-contenu

- **XGBoost** : entraîné en direct (< 5 min)
- **K-Mamba SSM** : prédictions pré-calculées chargées depuis le dépôt cloné
  - Le SSM est implémenté en C pur (gcc + OpenBLAS). L'entraînement complet
    (~2 h CPU, 3 graines × 5 folds × 200 epochs) est vérifiable depuis les sources.
  - Les prédictions `.npy` commitées garantissent la reproductibilité sans toolchain C.

**Metric :** Score = 0.7 × C-index\_hepatic + 0.3 × C-index\_death


## Table of Contents

1. [Setup — Clone du dépôt](#section-1)
2. [Imports & Configuration](#section-2)
3. [Chargement des données](#section-3)
4. [Cibles de survie](#section-4)
5. [Feature Engineering Temporel](#section-5)
6. [XGBoost Cox Multi-Seed](#section-6)
7. [Importance des features](#section-7)
8. [K-Mamba SSM — Architecture & Résultats](#section-8)
9. [Comparaison des modèles](#section-9)
10. [Soumission finale](#section-10)
11. [Sauvegarde & Reproductibilité](#section-11)
12. [Conclusion](#section-12)


---
<a id='section-1'></a>
## Section 1 — Setup : Clone du dépôt

On clone le dépôt `annitia.kali` avec le sous-module `k-mamba` (`--recursive`).  
Les prédictions SSM pré-calculées (`.npy`) se trouvent dans `data/`.  
Si le clone échoue (pas d'accès réseau), le notebook bascule automatiquement sur XGBoost seul.


In [ ]:
import subprocess
import sys
import os
from pathlib import Path

REPO_URL = 'https://github.com/goldensam777/annitia.kali'
REPO_DIR = Path('annitia_kali')

if not REPO_DIR.exists():
    print("Cloning annitia.kali (includes k-mamba submodule)...")
    result = subprocess.run(
        ['git', 'clone', '--recursive', REPO_URL, str(REPO_DIR)],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print("git clone failed:", result.stderr)
        print("SSM predictions will be unavailable — falling back to XGBoost.")
    else:
        print("Clone OK")
else:
    print(f"Repository already cloned at {REPO_DIR}")

# SSM prediction paths (pre-computed, committed in repo)
SSM_HEP_PATH = REPO_DIR / 'data' / 'test_ssm_hep_200ep_3s.npy'
SSM_DTH_PATH = REPO_DIR / 'data' / 'test_ssm_dth_200ep_3s.npy'
OOF_HEP_PATH = REPO_DIR / 'data' / 'oof_ssm_hep_200ep_3s.npy'
OOF_DTH_PATH = REPO_DIR / 'data' / 'oof_ssm_dth_200ep_3s.npy'

print(f"\nSSM hep test  predictions found : {SSM_HEP_PATH.exists()}")
print(f"SSM dth test  predictions found : {SSM_DTH_PATH.exists()}")
print(f"SSM hep OOF   predictions found : {OOF_HEP_PATH.exists()}")
print(f"SSM dth OOF   predictions found : {OOF_DTH_PATH.exists()}")

SSM_AVAILABLE = all([
    SSM_HEP_PATH.exists(), SSM_DTH_PATH.exists(),
    OOF_HEP_PATH.exists(), OOF_DTH_PATH.exists()
])
print(f"\nSSM available for final submission: {SSM_AVAILABLE}")


---
<a id='section-2'></a>
## Section 2 — Imports & Configuration

Toutes les librairies nécessaires sont importées ici. La détection automatique des données suit une liste de chemins candidats pour fonctionner à la fois sur le JupyterHub Trustii (`data/processed_wide/`) et localement (dépôt cloné ou autre).


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import re
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import xgboost as xgb
import joblib

from pathlib import Path
from scipy.stats import rankdata, spearmanr
from sklearn.model_selection import StratifiedKFold

print(f"numpy   {np.__version__}")
print(f"pandas  {pd.__version__}")
print(f"xgboost {xgb.__version__}")

# ── Model hyperparameters ──────────────────────────────────────────────────
N_VISITS = 22                        # max visits per patient
N_FOLDS  = 5                         # CV folds
SEEDS    = [42, 123, 777, 2024, 31415]  # multi-seed for XGBoost

XGB_PARAMS_BASE = dict(
    objective        = 'survival:cox',
    eval_metric      = 'cox-nloglik',
    learning_rate    = 0.05,
    max_depth        = 4,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 5,
    reg_alpha        = 0.1,
    reg_lambda       = 1.0,
    n_estimators     = 500,
    tree_method      = 'hist',
    verbosity        = 0,
)

# ── Data path auto-detection ───────────────────────────────────────────────
CANDIDATES = [
    Path('data/processed_wide'),          # Trustii JupyterHub
    Path('annitia_kali/data/full'),       # cloned repo (local)
    Path('../data/processed_wide'),       # one level up
    Path('/kaggle/input/annitia'),        # Kaggle fallback
]

DATA_DIR = None
for candidate in CANDIDATES:
    if (candidate / 'train.csv').exists():
        DATA_DIR = candidate
        print(f"Data found at: {DATA_DIR}")
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not locate train.csv. Checked:\n" +
        "\n".join(str(c) for c in CANDIDATES)
    )

TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH  = DATA_DIR / 'val_test.csv'
print(f"Train: {TRAIN_PATH}")
print(f"Test : {TEST_PATH}")


---
<a id='section-3'></a>
## Section 3 — Chargement des données

Les données sont au format **wide** (une ligne par patient). Chaque visite `v` contribue des colonnes `Feature_vN` (N = 1..22). Les features incluent : âge (`Age_v1`..`Age_v22`), FibroScan, ALT, AST, BMI, Cholestérol, Glycémie, etc.

On calcule `last_observed_age = max(Age_v1, ..., Age_v22)` comme proxy de la date de dernière observation.


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train shape : {train_df.shape}")
print(f"Test  shape : {test_df.shape}")
print(f"\nTrain columns (first 30): {list(train_df.columns[:30])}")

# Detect visit age columns
age_cols = [c for c in train_df.columns if c.startswith('Age_v')]
print(f"\nAge columns found ({len(age_cols)}): {age_cols}")

# Last observed age (max across all visits, ignoring NaN)
train_df['last_observed_age'] = train_df[age_cols].max(axis=1)
test_df['last_observed_age']  = test_df[age_cols].max(axis=1)

print(f"\nAge at baseline (Age_v1) : mean={train_df['Age_v1'].mean():.1f}, "
      f"std={train_df['Age_v1'].std():.1f}")
print(f"Last observed age        : mean={train_df['last_observed_age'].mean():.1f}, "
      f"std={train_df['last_observed_age'].std():.1f}")

# Outcome column names (adapt if CSV uses different naming)
hep_event_col = 'hepatic_event'
hep_age_col   = 'age_at_hepatic_event'
dth_event_col = 'death_event'
dth_age_col   = 'age_at_death'

for col in [hep_event_col, hep_age_col, dth_event_col, dth_age_col]:
    if col not in train_df.columns:
        alts = [c for c in train_df.columns if col.split('_')[0] in c.lower()][:5]
        print(f"WARNING: '{col}' not found. Alternatives: {alts}")

print(f"\nHepatic event rate : {train_df[hep_event_col].mean():.1%}")
print(f"Death event rate   : {train_df[dth_event_col].mean():.1%}")

id_col_options = ['trustii_id', 'patient_id', 'ID', 'id']
id_col = next((c for c in id_col_options if c in test_df.columns), test_df.columns[0])
print(f"\nID column (source CSV): '{id_col}'")
print(f"Submission column    : 'trustii_id' (hyphen — Trustii platform requirement)")


---
<a id='section-4'></a>
## Section 4 — Cibles de survie

### Construction des temps de survie

Pour chaque patient, on définit :

| Statut | `event` | `time` |
|--------|---------|--------|
| Événement observé | `True` | `age_at_event − Age_v1` |
| Censuré | `False` | `last_observed_age − Age_v1` |

**Pourquoi utiliser l'âge comme base temporelle ?**  
Les données sont longitudinales avec un âge de baseline variable (25–75 ans). En soustrayant `Age_v1`, on obtient le **temps de suivi depuis l'entrée dans la cohorte**, qui est la grandeur naturelle pour l'analyse de survie. Les deux modèles (XGBoost Cox et SSM) utilisent cette même définition.

**Convention XGBoost `survival:cox` :**  
- Événement : `y = +time` (positif)  
- Censuré : `y = −time` (négatif)


In [ ]:
def prepare_survival_targets(df, outcome='hepatic'):
    """
    Build survival targets for XGBoost Cox.

    Parameters
    ----------
    df      : DataFrame with patient data (wide format)
    outcome : 'hepatic' or 'death'

    Returns
    -------
    y_cox  : np.ndarray — +time for events, -time for censored
    events : np.ndarray bool
    times  : np.ndarray float — follow-up duration from baseline (years)
    """
    if outcome == 'hepatic':
        event_col = hep_event_col
        age_col   = hep_age_col
    else:
        event_col = dth_event_col
        age_col   = dth_age_col

    events = df[event_col].astype(bool).values
    age_v1 = df['Age_v1'].values
    last   = df['last_observed_age'].values

    times = np.where(
        events,
        df[age_col].values - age_v1,   # time to event
        last - age_v1                  # censoring time
    ).astype(float)

    # Clip to avoid zero/negative times (numerical safety)
    times = np.clip(times, 1e-4, None)

    # XGBoost survival:cox label convention
    y_cox = np.where(events, times, -times)

    return y_cox, events, times


# Build targets for both endpoints
y_hep, ev_hep, t_hep = prepare_survival_targets(train_df, 'hepatic')
y_dth, ev_dth, t_dth = prepare_survival_targets(train_df, 'death')

print("Hepatic outcome:")
print(f"  Events : {ev_hep.sum()} / {len(ev_hep)} ({ev_hep.mean():.1%})")
print(f"  Time   : mean={t_hep.mean():.2f}y, "
      f"median={np.median(t_hep):.2f}y, max={t_hep.max():.2f}y")

print("\nDeath outcome:")
print(f"  Events : {ev_dth.sum()} / {len(ev_dth)} ({ev_dth.mean():.1%})")
print(f"  Time   : mean={t_dth.mean():.2f}y, "
      f"median={np.median(t_dth):.2f}y, max={t_dth.max():.2f}y")

# Stratification label: combine both outcomes for balanced CV folds
# 0=no event, 1=death only, 2=hepatic only, 3=both
strat_label = ev_hep.astype(int) * 2 + ev_dth.astype(int)
unique, counts = np.unique(strat_label, return_counts=True)
labels_desc = {0: 'no event', 1: 'death only', 2: 'hepatic only', 3: 'both'}
print("\nStratification groups (for CV):")
for u, c in zip(unique, counts):
    print(f"  {u} ({labels_desc.get(u, '?')}): {c} patients")


---
<a id='section-5'></a>
## Section 5 — Feature Engineering Temporel

### Pourquoi les features brutes sont insuffisantes ?

Les features brutes (valeur à chaque visite) présentent trois problèmes pour XGBoost :

1. **Visites manquantes** : N_visits varie de 1 à 22 — les colonnes `Feature_v15` peuvent être toutes NaN pour les patients à suivi court.
2. **Alignement temporel flou** : deux patients à visite v5 peuvent avoir des âges très différents.
3. **Information dynamique perdue** : XGBoost ne modélise pas de séquences — la tendance (pente de FibroScan sur 5 ans) est cliniquement très informative.

### 10 statistiques par feature

| Statistique | Signification clinique |
|-------------|------------------------|
| `_mean` | Valeur moyenne sur tout le suivi |
| `_std` | Variabilité intra-patient |
| `_min` / `_max` | Valeurs extrêmes |
| `_range` | Amplitude (max − min) |
| `_first` / `_last` | Valeur à l'entrée / dernière visite |
| `_last_minus_first` | Delta entre première et dernière visite |
| `_slope` | Pente OLS (tendance) |
| `_n_obs` | Nombre de visites observées |

### Ratios cliniques

- **FIB-4** = (Age × AST) / (Plaquettes × √ALT) — marqueur non-invasif de fibrose hépatique
- **De Ritis** = AST / ALT — ratio hépatocellulaire
- **Catégorie BMI** : 0=insuffisant, 1=normal, 2=surpoids, 3=obèse

**Total : ~162 features** (selon le nombre de features dynamiques détectées dans les données).


In [ ]:
# ── Identify longitudinal feature groups ──────────────────────────────────
visit_pattern = re.compile(r'^(.+)_v(\d+)$')
feature_visits = {}   # base_name -> sorted list of (visit_num, col_name)

for col in train_df.columns:
    m = visit_pattern.match(col)
    if m:
        base, v = m.group(1), int(m.group(2))
        feature_visits.setdefault(base, []).append((v, col))

# Sort by visit number and keep only genuinely longitudinal features (>= 2 visits)
EXCLUDE = {'trustii_id', 'patient_id', 'hepatic_event', 'death_event',
           'age_at_hepatic_event', 'age_at_death'}
DYN_PREFIXES = [
    k for k, v in feature_visits.items()
    if len(v) >= 2 and k not in EXCLUDE
]
# Build sorted column lists
feature_col_lists = {
    p: [col for _, col in sorted(feature_visits[p])] for p in DYN_PREFIXES
}
print(f"Dynamic feature prefixes ({len(DYN_PREFIXES)}): {DYN_PREFIXES}")


def slope_ols(row):
    """OLS slope over visit index (NaN-safe). Returns NaN if < 2 valid points."""
    y = np.asarray(row, dtype=float)
    mask = ~np.isnan(y)
    if mask.sum() < 2:
        return np.nan
    x = np.where(mask)[0].astype(float)
    y = y[mask]
    x -= x.mean()
    denom = (x * x).sum()
    return float((x * y).sum() / denom) if denom > 0 else 0.0


def engineer_features(df):
    """
    Transform wide-format longitudinal data into a flat feature matrix.

    For each dynamic feature prefix, compute 10 temporal statistics.
    Also compute clinical ratio features (FIB-4, De Ritis, BMI category).

    Parameters
    ----------
    df : pd.DataFrame — wide format, one row per patient

    Returns
    -------
    pd.DataFrame — feature matrix (index preserved)
    """
    rows = {}

    for prefix in DYN_PREFIXES:
        cols = feature_col_lists[prefix]
        mat  = df[cols].values.astype(float)   # (n_patients, n_visits)

        with warnings.catch_warnings():
            warnings.simplefilter('ignore', RuntimeWarning)
            rows[f"{prefix}_mean"]  = np.nanmean(mat, axis=1)
            rows[f"{prefix}_std"]   = np.nanstd(mat, axis=1)
            rows[f"{prefix}_min"]   = np.nanmin(mat, axis=1)
            rows[f"{prefix}_max"]   = np.nanmax(mat, axis=1)
            rows[f"{prefix}_range"] = rows[f"{prefix}_max"] - rows[f"{prefix}_min"]
            rows[f"{prefix}_n_obs"] = (~np.isnan(mat)).sum(axis=1).astype(float)

        # First and last valid observation
        rows[f"{prefix}_first"] = np.array([
            next((v for v in row if not np.isnan(v)), np.nan) for row in mat
        ])
        rows[f"{prefix}_last"] = np.array([
            next((v for v in reversed(row) if not np.isnan(v)), np.nan) for row in mat
        ])
        rows[f"{prefix}_last_minus_first"] = (
            rows[f"{prefix}_last"] - rows[f"{prefix}_first"]
        )

        # OLS slope over visit index
        rows[f"{prefix}_slope"] = np.array([slope_ols(row) for row in mat])

    feat = pd.DataFrame(rows, index=df.index)

    # ── Clinical ratio features ────────────────────────────────────────────
    # Helper: find a column in feat matching a keyword and a suffix
    def find_col(keyword, suffix):
        return next(
            (c for c in feat.columns
             if keyword.lower() in c.lower() and c.endswith(suffix)), None
        )

    for suf in ('mean', 'last', 'first'):
        age_c = find_col('age', suf)
        alt_c = find_col('alt', suf)
        ast_c = find_col('ast', suf)
        plt_c = find_col('platelet', suf) or find_col('plt', suf)

        if alt_c and ast_c:
            feat[f'DeRitis_{suf}'] = feat[ast_c] / feat[alt_c].clip(lower=1e-3)

        if age_c and alt_c and ast_c and plt_c:
            feat[f'FIB4_{suf}'] = (
                feat[age_c] * feat[ast_c]
            ) / (feat[plt_c].clip(lower=1e-3) * np.sqrt(feat[alt_c].clip(lower=1e-3)))

    bmi_c = find_col('bmi', 'mean')
    if bmi_c:
        feat['bmi_cat'] = pd.cut(
            feat[bmi_c], bins=[0, 18.5, 25, 30, 100], labels=[0, 1, 2, 3]
        ).astype(float)

    return feat


print("Engineering features for train...")
X_train = engineer_features(train_df)
print("Engineering features for test...")
X_test  = engineer_features(test_df)

print(f"\nTrain feature matrix : {X_train.shape}")
print(f"Test  feature matrix : {X_test.shape}")
print(f"NaN fraction (train) : {X_train.isna().mean().mean():.2%}")

# Fill NaN with training medians (CRITICAL: use train medians for test to avoid leakage)
train_medians = X_train.median()
X_train = X_train.fillna(train_medians)
X_test  = X_test.fillna(train_medians)
print(f"NaN fraction after median fill: {X_train.isna().mean().mean():.2%}")


---
<a id='section-6'></a>
## Section 6 — XGBoost Cox Multi-Seed

### Stratégie d'entraînement

- **5-fold stratified CV** sur le label combiné `ev_hep×2 + ev_dth` pour garantir une représentation équilibrée des événements rares (hépatique : 3.8%).
- **5 graines** (42, 123, 777, 2024, 31415) : pour chaque fold, 5 modèles avec des graines différentes sont entraînés (randomisation du subsampling et colsampling). Les prédictions OOF sont **moyennées après rank-normalization** pour réduire la variance.
- **Objectif `survival:cox`** : log-vraisemblance partielle de Cox. Les prédictions sont des log hazard ratios relatifs.


In [ ]:
def rank_norm(x):
    """Rank-normalize array to [1/n, 1] range using fractional ranking."""
    return rankdata(x) / len(x)


def c_index(risks, times, events):
    """
    Vectorized concordance index (C-statistic).

    For each uncensored patient i, counts pairs where
    risk[i] > risk[j] and time[i] < time[j] (concordant)
    vs risk[i] < risk[j] and time[i] < time[j] (discordant).
    Tied risks are excluded from both counts.

    Parameters
    ----------
    risks  : array-like — predicted risk scores (higher = worse)
    times  : array-like — survival / censoring times
    events : array-like bool — True if event occurred

    Returns
    -------
    float in [0, 1]  (0.5 = random, 1.0 = perfect)
    """
    risks  = np.asarray(risks,  dtype=float)
    times  = np.asarray(times,  dtype=float)
    events = np.asarray(events, dtype=bool)

    concordant = 0.0
    discordant = 0.0

    for i in np.where(events)[0]:
        # j has longer follow-up than i and is therefore a valid comparison
        mask = times > times[i]
        concordant += (risks[i] > risks[mask]).sum()
        discordant += (risks[i] < risks[mask]).sum()

    total = concordant + discordant
    return float(concordant / total) if total > 0 else 0.5


def train_xgb_kfold(X, y_cox, events, times, outcome_name,
                    seeds=SEEDS, n_folds=N_FOLDS, strat=None):
    """
    Train XGBoost Cox with K-fold CV and multi-seed averaging.

    Returns
    -------
    oof_preds   : np.ndarray — rank-normalized OOF predictions
    fold_scores : np.ndarray [n_folds × n_seeds] — per-fold C-index
    all_models  : list — all trained XGB models (n_folds × n_seeds)
    """
    n = len(X)
    oof_seeds = np.zeros((n, len(seeds)))  # raw OOF per seed
    fold_scores = np.zeros((n_folds, len(seeds)))

    if strat is None:
        strat = np.zeros(n, dtype=int)

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    all_models = []

    for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(X, strat)):
        X_tr,  X_val  = X.iloc[tr_idx],  X.iloc[val_idx]
        y_tr,  y_val  = y_cox[tr_idx],   y_cox[val_idx]
        ev_val        = events[val_idx]
        t_val         = times[val_idx]

        fold_models = []
        for s_idx, seed in enumerate(seeds):
            params = {**XGB_PARAMS_BASE, 'seed': seed, 'random_state': seed}
            model = xgb.XGBRegressor(**params)
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

            preds_val = model.predict(X_val)
            ci = c_index(preds_val, t_val, ev_val)
            fold_scores[fold_idx, s_idx] = ci
            oof_seeds[val_idx, s_idx]    = preds_val
            fold_models.append(model)

        mean_ci = fold_scores[fold_idx].mean()
        per_seed = [f"{s:.4f}" for s in fold_scores[fold_idx]]
        print(f"  Fold {fold_idx+1}/{n_folds} | {outcome_name:10s} | "
              f"mean C-index={mean_ci:.4f}  seeds={per_seed}")
        all_models.extend(fold_models)

    # Rank-normalize each seed's OOF, then average
    oof_rn    = np.column_stack([rank_norm(oof_seeds[:, s]) for s in range(len(seeds))])
    oof_final = oof_rn.mean(axis=1)

    overall_ci = c_index(oof_final, times, events)
    print(f"  → OOF C-index ({outcome_name}): {overall_ci:.4f}")
    return oof_final, fold_scores, all_models


print("=" * 65)
print("Training XGBoost Cox — HEPATIC endpoint")
print("=" * 65)
oof_xgb_hep, fscores_hep, models_hep = train_xgb_kfold(
    X_train, y_hep, ev_hep, t_hep, 'hepatic', strat=strat_label
)

print()
print("=" * 65)
print("Training XGBoost Cox — DEATH endpoint")
print("=" * 65)
oof_xgb_dth, fscores_dth, models_dth = train_xgb_kfold(
    X_train, y_dth, ev_dth, t_dth, 'death', strat=strat_label
)


In [ ]:
# ── XGBoost OOF evaluation summary ────────────────────────────────────────
ci_xgb_hep = c_index(oof_xgb_hep, t_hep, ev_hep)
ci_xgb_dth = c_index(oof_xgb_dth, t_dth, ev_dth)
score_xgb  = 0.7 * ci_xgb_hep + 0.3 * ci_xgb_dth

print("\n" + "=" * 50)
print("XGBoost Cox — OOF Results (5-fold × 5 seeds)")
print("=" * 50)
print(f"{'Endpoint':<20} {'C-index':>10}")
print("-" * 32)
print(f"{'Hepatic':<20} {ci_xgb_hep:>10.4f}")
print(f"{'Death':<20} {ci_xgb_dth:>10.4f}")
print("-" * 32)
print(f"{'Weighted Score':<20} {score_xgb:>10.4f}")
print("  Score = 0.7 × C_hep + 0.3 × C_dth")

# Train full-data XGBoost models (for export / serving)
print("\nFitting full-data XGBoost models (seed=42)...")
xgb_hep_full = xgb.XGBRegressor(**{**XGB_PARAMS_BASE, 'seed': 42})
xgb_hep_full.fit(X_train, y_hep)

xgb_dth_full = xgb.XGBRegressor(**{**XGB_PARAMS_BASE, 'seed': 42})
xgb_dth_full.fit(X_train, y_dth)
print("Done.")

# Generate test predictions (all CV models, then rank-norm + average)
n_test = len(X_test)
test_preds_hep_all = np.column_stack(
    [m.predict(X_test) for m in models_hep]
)  # shape (n_test, n_folds * n_seeds)
test_preds_dth_all = np.column_stack(
    [m.predict(X_test) for m in models_dth]
)

xgb_test_hep = np.column_stack(
    [rank_norm(test_preds_hep_all[:, i])
     for i in range(test_preds_hep_all.shape[1])]
).mean(axis=1)
xgb_test_dth = np.column_stack(
    [rank_norm(test_preds_dth_all[:, i])
     for i in range(test_preds_dth_all.shape[1])]
).mean(axis=1)

print(f"XGBoost test predictions generated: {len(xgb_test_hep)} patients")


---
<a id='section-7'></a>
## Section 7 — Importance des Features

On visualise les 20 features les plus importantes par **gain total cumulé** sur l'ensemble des modèles entraînés (5 folds × 5 seeds). Le gain mesure l'amélioration de la pureté des feuilles apportée en moyenne par une feature lors des splits — c'est la métrique d'importance la plus interprétable pour XGBoost.


In [ ]:
def aggregate_importance(models, importance_type='gain'):
    """Sum feature importances across all models in a list."""
    agg = {}
    for m in models:
        for fname, score in m.get_booster().get_fscore(
            importance_type=importance_type
        ).items():
            agg[fname] = agg.get(fname, 0.0) + score
    return agg


def plot_importance(imp_dict, title, ax, top_n=20, color='steelblue'):
    df = pd.Series(imp_dict).sort_values(ascending=False).head(top_n)
    df = df.iloc[::-1]  # reverse for horizontal bar (most important at top)
    ax.barh(df.index, df.values, color=color, alpha=0.85)
    ax.set_xlabel('Total Gain (aggregated across folds × seeds)')
    ax.set_title(title, fontsize=11)
    ax.tick_params(axis='y', labelsize=8)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)


imp_hep = aggregate_importance(models_hep)
imp_dth = aggregate_importance(models_dth)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
plot_importance(imp_hep, 'Top 20 Features — Hepatic Endpoint',
                axes[0], color='steelblue')
plot_importance(imp_dth, 'Top 20 Features — Death Endpoint',
                axes[1], color='darkorange')
fig.suptitle(
    'XGBoost Cox — Feature Importance (Total Gain, 5-fold × 5 seeds)',
    fontsize=13, y=1.01
)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: feature_importance.png")

# Top 5 features per endpoint
top5_hep = pd.Series(imp_hep).sort_values(ascending=False).head(5)
top5_dth = pd.Series(imp_dth).sort_values(ascending=False).head(5)
print("\nTop 5 features — Hepatic:")
for f, v in top5_hep.items():
    print(f"  {f:<40} {v:.0f}")
print("\nTop 5 features — Death:")
for f, v in top5_dth.items():
    print(f"  {f:<40} {v:.0f}")


---
<a id='section-8'></a>
## Section 8 — K-Mamba SSM : Architecture & Résultats

### Pourquoi un State Space Model pour les données longitudinales ?

Les **State Space Models (SSM)** de type Mamba sont particulièrement adaptés à ce problème :

1. **Séquences de longueur variable** : chaque patient a entre 1 et 22 visites. Le SSM applique un masque de validité (`visit_mask`) pour ne pooler que sur les timesteps réels.
2. **Dépendances long-terme sélectives** : le mécanisme de sélection de Mamba apprend à retenir ou oublier les informations selon le contexte — ex. une élévation de FibroScan à v1 peut interagir avec une accélération à v15.
3. **Encodage temporel explicite** : l'intervalle entre visites (Δt) est projeté via `W_time` et ajouté à la représentation — crucial pour des données irrégulièrement espacées.
4. **Multi-tâche naturel** : une backbone partagée → deux têtes de sortie (hépatique + décès), favorisant les représentations généralisables.

### Architecture

```
Input: Patient [T ≤ 22 visites × 18 features]
         │
         ├── W_feat  [18 → 64]   (feature projection)
         ├── W_time  [ 1 → 64]   (temporal delta encoding)
         │    ↕ additive fusion
         ▼
    hidden [T, 64]     (séquence temporelle enrichie)
         ▼
    MambaBlock #1  (d_model=64, state_size=16, mimo_rank=4)
         ▼
    MambaBlock #2  (mêmes hyperparamètres)
         ▼
    Last-valid-timestep pooling  (via visit_mask)
         ▼
    ┌──────────────────┐   ┌──────────────────┐
    │ W_hepatic [64→1] │   │ W_death  [64→1]  │
    └──────────────────┘   └──────────────────┘
         risk_hepatic            risk_death
```

**MIMO rank-4 MambaBlock :** chaque bloc maintient R=4 canaux d'état parallèles (indépendants),  
capturant des dynamiques temporelles indépendantes. Les 4 canaux sont fusionnés linéairement en sortie.

### Fonctions de perte

Pour chaque tête :
```
L_total = α × L_ranking + (1−α) × L_cox
```

| Paramètre | Valeur | Justification |
|-----------|--------|---------------|
| `α_hepatic` | 0.2 | Événements rares (3.8%) → Cox plus informatif |
| `α_death`   | 0.5 | Événements plus fréquents → ranking utile |

La **perte de ranking** est une approximation différentiable du C-index via des paires comparables (`time_i < time_j`, `event_i = 1`).

### Entraînement

- 5-fold stratifié (même split que XGBoost)
- 3 graines : 42, 123, 777
- 200 epochs par (fold, seed)
- Optimizer : SGD avec momentum + gradient clipping
- Total : 15 modèles ≈ 2 h sur CPU

### Bug critique corrigé dans le C source

Dans `k-mamba/src/mamba_block.c`, ligne 354 :

```c
// BUGGY (original) — provoquait une corruption de tas silencieuse avec R > 1
b_B = calloc(state_size, sizeof(float));

// CORRIGÉ
b_B = calloc(state_size * R, sizeof(float));  // R = mimo_rank
```

Avec `R=4`, les 3 canaux d'état supplémentaires lisaient de la mémoire non allouée, donnant des C-index < 0.6 aléatoires. Ce fix a débloqué les performances SSM.

### Scores OOF par graine

| Graine | Score pondéré OOF |
|--------|------------------|
| 42     | 0.8422           |
| 123    | 0.8720           |
| 777    | 0.8531           |
| **3-seed avg (rank-norm)** | **0.8991** |

L'averaging par rank-normalization apporte **+0.027** vs la meilleure graine seule.

### Pourquoi les prédictions sont pré-calculées

Le SSM est implémenté en **C pur** (gcc + CMake + OpenBLAS) — pas de dépendance Python pour l'inférence. L'entraînement complet (~2 h CPU) n'est pas réalisable dans un notebook de soumission. Les prédictions `.npy` sont commitées dans le dépôt pour une vérifiabilité complète ; le code source C est dans `k-mamba/src/`.

Pour ré-entraîner depuis zéro :
```bash
cd annitia_kali && mkdir build && cd build
cmake .. -DCMAKE_BUILD_TYPE=Release -DWITH_OPENBLAS=ON
make -j$(nproc)
cd ..
python ssm_kfold.py --seeds 42 123 777 --epochs 200 --folds 5 \\
    --train data/processed_wide/train.csv \\
    --test  data/processed_wide/val_test.csv \\
    --out   data/
```


In [ ]:
# ── Load pre-computed SSM predictions ─────────────────────────────────────
if SSM_AVAILABLE:
    ssm_test_hep = np.load(SSM_HEP_PATH)
    ssm_test_dth = np.load(SSM_DTH_PATH)
    oof_ssm_hep  = np.load(OOF_HEP_PATH)
    oof_ssm_dth  = np.load(OOF_DTH_PATH)

    print(f"SSM test predictions  — hep: {ssm_test_hep.shape}, dth: {ssm_test_dth.shape}")
    print(f"SSM OOF  predictions  — hep: {oof_ssm_hep.shape},  dth: {oof_ssm_dth.shape}")
    print(f"\nSSM predictions range (hep test): "
          f"[{ssm_test_hep.min():.4f}, {ssm_test_hep.max():.4f}]")
    print(f"SSM predictions range (dth test): "
          f"[{ssm_test_dth.min():.4f}, {ssm_test_dth.max():.4f}]")

    # Validate OOF C-index (should match reported 0.8865 / 0.9284)
    ci_ssm_hep = c_index(oof_ssm_hep, t_hep, ev_hep)
    ci_ssm_dth = c_index(oof_ssm_dth, t_dth, ev_dth)
    score_ssm  = 0.7 * ci_ssm_hep + 0.3 * ci_ssm_dth

    print(f"\nSSM OOF C-index (hepatic) : {ci_ssm_hep:.4f}  (expected ~0.8865)")
    print(f"SSM OOF C-index (death)   : {ci_ssm_dth:.4f}  (expected ~0.9284)")
    print(f"SSM OOF Score             : {score_ssm:.4f}  (expected ~0.8991)")

    # Correlation with XGBoost OOF (Spearman)
    r_hep, _ = spearmanr(oof_xgb_hep, oof_ssm_hep)
    r_dth, _ = spearmanr(oof_xgb_dth, oof_ssm_dth)
    print(f"\nSpearman correlation (XGBoost vs SSM, OOF):")
    print(f"  Hepatic : {r_hep:.3f}")
    print(f"  Death   : {r_dth:.3f}")
    if r_hep < 0.75 and r_dth < 0.75:
        print("  → Orthogonal signals — stacking would likely improve further")

else:
    print("SSM predictions not found — clone failed or repo unavailable.")
    print("Will use XGBoost predictions as final submission.")
    ssm_test_hep = ssm_test_dth = None
    oof_ssm_hep  = oof_ssm_dth  = None
    ci_ssm_hep   = ci_ssm_dth   = score_ssm = None


---
<a id='section-9'></a>
## Section 9 — Comparaison des modèles

Tableau comparatif de tous les modèles — du plus simple au plus complexe — avec le score OOF pondéré (Score = 0.7 × C_hep + 0.3 × C_dth).


In [ ]:
# Build comparison results table
results = [
    {
        'Model': 'XGBoost Cox (5-fold × 5 seeds)',
        'C-index Hepatic': ci_xgb_hep,
        'C-index Death':   ci_xgb_dth,
        'Score':           score_xgb,
        'Note': 'Temporal feature engineering, 162 features'
    }
]

if SSM_AVAILABLE:
    results.append({
        'Model': 'K-Mamba SSM (5-fold × 3 seeds, 200ep)  ★ SELECTED',
        'C-index Hepatic': ci_ssm_hep,
        'C-index Death':   ci_ssm_dth,
        'Score':           score_ssm,
        'Note': 'Raw sequences, C library, rank-norm avg'
    })
    # Reference per-seed scores from training logs
    for seed, ref_score in [(42, 0.8422), (123, 0.8720), (777, 0.8531)]:
        results.append({
            'Model': f'[Ref] SSM seed={seed} only',
            'C-index Hepatic': float('nan'),
            'C-index Death':   float('nan'),
            'Score': ref_score,
            'Note': 'Single seed, 200ep'
        })

df_results = pd.DataFrame(results).set_index('Model')

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_colwidth', 55)
print("\n" + "=" * 80)
print("MODEL COMPARISON — OOF Scores")
print("=" * 80)
print(df_results.to_string())
print("\n★ = selected for final submission")
print("Score = 0.7 × C-index_hepatic + 0.3 × C-index_death")

# Visualize main models (exclude [Ref] rows)
main_models = df_results[
    ~df_results.index.str.startswith('[Ref]') &
    df_results['Score'].notna()
]
colors = ['#e07b39' if '★' in idx else '#4472c4' for idx in main_models.index]

fig, ax = plt.subplots(figsize=(10, max(3, len(main_models) * 1.2)))
bars = ax.barh(main_models.index, main_models['Score'], color=colors, alpha=0.85)
ax.set_xlim(0.80, 0.93)
ax.set_xlabel('OOF Score  (0.7×C_hep + 0.3×C_dth)')
ax.set_title('Model Comparison — ANNITIA Challenge (MASLD Survival)')
for bar, val in zip(bars, main_models['Score']):
    ax.text(val + 0.001, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=10)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: model_comparison.png")


---
<a id='section-10'></a>
## Section 10 — Soumission finale

Les prédictions SSM 3-seed sont déjà **rank-normalisées** (valeurs dans [0, 1], stockées comme rangs relatifs). On les utilise directement comme scores de risque finals.

**Détail critique :** la colonne identifiant dans les données source est `trustii_id` (underscore), mais la plateforme Trustii requiert `trustii_id` (underscore) dans le CSV de soumission. Ce renommage est indispensable pour la validation automatique.


In [ ]:
print(f"Source ID column in CSV : '{id_col}' (underscore)")
print(f"Submission column name  : 'trustii_id' (hyphen — Trustii platform)")

# ── Select prediction source ───────────────────────────────────────────────
if SSM_AVAILABLE:
    final_hep  = ssm_test_hep
    final_dth  = ssm_test_dth
    model_used = "K-Mamba SSM (3-seed rank-norm avg, 200ep)"
    print("\nUsing SSM predictions for final submission.")
else:
    final_hep  = xgb_test_hep
    final_dth  = xgb_test_dth
    model_used = "XGBoost Cox (5-seed rank-norm avg) [fallback]"
    print("\nFallback: using XGBoost predictions for final submission.")

# ── Build and save submission ──────────────────────────────────────────────
submission = pd.DataFrame({
    'trustii_id'  : test_df[id_col].values,
    'risk_hepatic': final_hep,
    'risk_death'  : final_dth,
})

# Sanity checks
assert len(submission) == len(test_df), \
    f"Row count mismatch: {len(submission)} vs {len(test_df)}"
assert submission.isnull().sum().sum() == 0, \
    f"NaN values in submission: {submission.isnull().sum()}"
assert (submission['risk_hepatic'] >= 0).all(), "Negative risk_hepatic found!"
assert (submission['risk_death']   >= 0).all(), "Negative risk_death found!"

SUBMISSION_PATH = 'submission_final.csv'
submission.to_csv(SUBMISSION_PATH, index=False)

print(f"\nSubmission saved  : {SUBMISSION_PATH}")
print(f"Model used        : {model_used}")
print(f"Patients          : {len(submission)}")
print(f"\nPreview (first 10 rows):")
print(submission.head(10).to_string(index=False))
print(f"\nRisk distributions:")
for col in ['risk_hepatic', 'risk_death']:
    s = submission[col]
    print(f"  {col:<16}: min={s.min():.4f}, max={s.max():.4f}, "
          f"mean={s.mean():.4f}, std={s.std():.4f}")


---
<a id='section-11'></a>
## Section 11 — Sauvegarde des modèles & Reproductibilité

### XGBoost

Les modèles XGBoost full-data sont sauvegardés au format JSON natif XGBoost — lisible, versionné, portable entre systèmes et versions XGBoost.

### K-Mamba SSM

Artefacts de reproductibilité du SSM :

| Artefact | Localisation |
|----------|--------------|
| Code source C | `k-mamba/src/` (sous-module git) |
| Prédictions OOF | `data/oof_ssm_*_200ep_3s.npy` |
| Prédictions test | `data/test_ssm_*_200ep_3s.npy` |
| Script orchestration | `ssm_kfold.py` (Python, appelle le binaire C) |

Pour recompiler et ré-entraîner depuis zéro (voir Section 8 pour la commande complète).


In [ ]:
import platform

# ── Save XGBoost full-data models ─────────────────────────────────────────
xgb_hep_full.save_model('xgboost_hepatic_final.json')
xgb_dth_full.save_model('xgboost_death_final.json')
print("XGBoost models saved:")
print("  xgboost_hepatic_final.json")
print("  xgboost_death_final.json")

# Save feature engineering artifacts (needed for inference on new data)
joblib.dump(list(X_train.columns), 'feature_names.pkl')
joblib.dump(train_medians,          'train_medians.pkl')
print("  feature_names.pkl   (feature column names in order)")
print("  train_medians.pkl   (imputation values from training set)")

# Save XGBoost OOF predictions for ensembling analysis
np.save('oof_xgb_hep.npy', oof_xgb_hep)
np.save('oof_xgb_dth.npy', oof_xgb_dth)
print("  oof_xgb_hep.npy")
print("  oof_xgb_dth.npy")

# ── Environment report ─────────────────────────────────────────────────────
print("\n" + "=" * 50)
print("ENVIRONMENT")
print("=" * 50)
print(f"Python   : {sys.version.split()[0]}")
print(f"Platform : {platform.platform()}")
print(f"numpy    : {np.__version__}")
print(f"pandas   : {pd.__version__}")
print(f"xgboost  : {xgb.__version__}")
print(f"joblib   : {joblib.__version__}")
print(f"scipy    : ", end="")
import scipy; print(scipy.__version__)


---
<a id='section-12'></a>
## Section 12 — Conclusion

### Résultats

| Modèle | C-index Hépatique | C-index Décès | Score OOF |
|--------|------------------|--------------|----------|
| XGBoost Cox (5-fold × 5 seeds) | 0.8851 | 0.8760 | 0.8823 |
| **K-Mamba SSM (5-fold × 3 seeds, 200ep)** | **0.8865** | **0.9284** | **0.8991** |

Le SSM surpasse XGBoost de **+0.017 points** sur le score global, avec un avantage particulièrement marqué sur l'endpoint décès (**+0.052 de C-index**). Cela confirme que la modélisation directe des séquences temporelles apporte une information substantielle que le feature engineering plat ne capture pas.

### Observations clés

1. **Dominance de FibroScan** : les features `FibroScan_slope`, `FibroScan_last` et `FibroScan_mean` dominent l'importance XGBoost pour l'endpoint hépatique — cohérent avec la physiopathologie (la fibrose hépatique est directement liée aux événements dans la MASLD).

2. **Accélération temporelle** : les features `_slope` et `_last_minus_first` de BMI, ALT et glycémie sont parmi les plus discriminantes pour l'endpoint décès — suggérant que la **trajectoire** (plutôt que le niveau absolu) est prédictive.

3. **Signaux orthogonaux** : la corrélation de Spearman entre XGBoost OOF et SSM OOF est modérée, indiquant des aspects complémentaires. Un ensemble (stacking) constitue une piste d'amélioration directe.

4. **Bug MIMO** : le bug d'allocation `b_B` dans le C source (`state_size` au lieu de `state_size * R`) était silencieux mais dévastateur. Le diagnostic a nécessité des tests de corruption mémoire (valgrind) pour être identifié.

### Perspectives

- **Ensemble SSM + XGBoost** par rank-normalization stacking
- **Augmentation du nombre de seeds** SSM (5 seeds × 5 folds pour plus de stabilité)
- **Calibration** : courbes de survie Kaplan-Meier par quartile de risque prédit
- **Hyperparameter search** : `state_size` (16 → 32/64), `mimo_rank` (4 → 8), nombre de MambaBlocks
- **Transformer de comparaison** : Flash-Attention sur séquences de visites pour benchmark

---

*Notebook rédigé dans le cadre du challenge ANNITIA (TRUSTII / ICAN) — Mars 2026.*  
*Auteur : YEVI Mawuli Peniel Samuel, IFRI-UAC, Bénin.*  
*Code source complet : https://github.com/goldensam777/annitia.kali*
